# Day 24 — EDA best practices
Objectives:
- Checklist-driven EDA.
- Summary stats, missingness, distributions, correlations.
- Clear, reproducible narrative in notebook form.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-24`. Read
`python/ds-60day/companion-guides/day24_eda_best_practices.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Exploratory data analysis (EDA) is a disciplined conversation with a
dataset, not a gallery of every possible chart. Start with an analytical
question, source/provenance, row grain, keys, and scope. Then inspect
data quality, individual distributions, relationships, segments, and
unusual records in an order that helps answer that question.

Separate an observation (“the median differs”) from a hypothesis (“one
segment may behave differently”) and from a causal claim, which EDA
alone normally cannot establish. Every table or chart needs a sentence
about evidence and a caveat. Missingness, duplicates, outliers, tiny
samples, and target leakage can make technically valid calculations
misleading.

### Vocabulary

- **EDA:** exploratory data analysis, structured investigation before formal conclusions.
- **provenance:** where data came from and under what conditions.
- **distribution:** the pattern of values, frequency, center, spread, and shape.
- **outlier:** an observation unusually distant under a stated context.
- **association:** a measured relationship that does not itself prove causation.
- **leakage:** information unavailable at the intended decision time contaminating analysis/modeling.

## Syntax anatomy

A useful EDA paragraph follows **question → method → observation →
limitation → next check**. `frame.describe(include="all")` can orient
you but is not a conclusion. A correlation matrix measures selected
pairwise associations under assumptions; it does not explain cause,
handle every nonlinear relationship, or protect against leakage.

### Worked example 1 — Create a bounded quality profile

Make basic risks visible before plotting relationships. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import pandas as pd

sample = pd.DataFrame({
    "customer_id": [1, 2, 2, 3],
    "amount": [10.0, 12.0, 12.0, None],
    "segment": ["new", "returning", "returning", "new"],
})
profile = {
    "shape": sample.shape,
    "duplicate_rows": int(sample.duplicated().sum()),
    "missing_rate": sample.isna().mean().round(2).to_dict(),
    "unique": sample.nunique(dropna=False).to_dict(),
}
profile

**Expected observation:** The profile reports four rows, one duplicate row, and a 0.25 missing rate for `amount`.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Compare robust and non-robust center

One extreme value affects the mean more than the median. Predict first; then run the next cell.

In [ ]:
values = pd.Series([10, 11, 12, 13, 200])
{"mean": values.mean(), "median": values.median(), "max": values.max()}

**Expected observation:** `{'mean': 49.2, 'median': 12.0, 'max': 200}`. The difference motivates inspection; it does not automatically justify deleting 200.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Write the question and row grain above the first calculation.
2. Profile missingness, duplicates, ranges, and key uniqueness before interpreting relationships.
3. Pair each finding with sample size, units, denominator, and a caveat.
4. Remove post-outcome or target-derived fields before correlation or predictive exploration.

**Alternative to compare:** Use a compact reusable profile for orientation, then write question-specific code rather than relying on a one-click profiling report.

**Boundary to test:** Constant/all-missing columns, tiny groups, extreme values, duplicated entities, Simpson's paradox, and time drift can invalidate naive summaries.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
df = sns.load_dataset('penguins').dropna()
df.describe(include='all')
sns.pairplot(df.select_dtypes('number'))
plt.show()
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='viridis')
plt.show()


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Produce a concise EDA for one local or already-cached dataset, organized as question → provenance/scope/grain → quality → univariate distributions → relationships/segments → findings/caveats.
   **Expected behavior:** every table/plot answers a written question and has an observation plus limitation. **Constraint:** avoid causal language and full-data dumps.
   **Verify:** restart and reproduce all results top to bottom.

2. Add a data-quality register with one row per issue: evidence/count, possible analytical impact, proposed treatment, validation check, and status. **Coverage:** missingness, duplicates/key uniqueness, ranges, categories, and at least one dataset-specific rule.
   **Verify:** trace how each accepted treatment changes row count or a key measure and preserve rejected/unresolved issues as caveats.

### Additional mastery practice

Organize exploratory data analysis around questions, grain, quality, and evidence. Separate observed patterns from hypotheses and causal claims.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict how one extreme value can change mean, median, standard deviation, and a scatterplot.
   **Progressive hint:** Robust and non-robust summaries respond differently to outliers.
   **Verify:** Compute statistics before/after adding the extreme value; record the exact mean/median/std changes and describe the visible plot-scale effect.
4. **Tracing:** Trace row grain from transaction-level data to a customer summary and explain which questions can no longer be answered afterward.
   **Progressive hint:** Aggregation discards within-customer event detail.
   **Verify:** List questions answerable at transaction grain, then assert the customer summary row count/uniqueness and identify at least one detail that cannot be recovered.
5. **Implementation:** Implement a compact profile returning shape, duplicate count, missing rates, numeric ranges, and unique counts.
   **Progressive hint:** Bound the result rather than dumping every row/value.
   **Verify:** Run the profile on ordinary, empty, duplicate, and missing fixtures; assert bounded keys/counts/rates without embedding full data values.
6. **Debugging:** Repair an EDA that calculates correlations after target-derived fields were added and treats the strongest coefficient as causal.
   **Progressive hint:** Remove leakage and label correlations as associations.
   **Verify:** Remove the target-derived field, recompute the association, and label it noncausal; assert the leakage column cannot enter the reported matrix.
7. **Edge case and explanation:** Handle constant, all-missing, and tiny-sample columns in plots and summaries; state which results are not meaningful.
   **Progressive hint:** A calculation returning a number does not guarantee interpretability.
   **Verify:** Detect constant/all-missing/tiny columns and assert each is skipped or annotated according to policy rather than reported as an interpretable statistic.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Produce a concise EDA for one local or already-cached dataset, organized as question → provenance/scope/grain → quality → univariate distributions → relationships/segments → findings/caveats. **Expected behavior:** every table/plot answers a written question and has an observation plus limitation. **Constraint:** avoid causal language and full-data dumps. **Verify:** restart and reproduce all results top to bottom.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Produce a concise EDA for one local or already-cached dataset, organized as question → provenance/scope/grain → quality → univariate distributions → relationships/segments → fin...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Add a data-quality register with one row per issue: evidence/count, possible analytical impact, proposed treatment, validation check, and status. **Coverage:** missingness, duplicates/key uniqueness, ranges, categories, and at least one dataset-specific rule. **Verify:** trace how each accepted treatment changes row count or a key measure and preserve rejected/unresolved issues as caveats.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Add a data-quality register with one row per issue: evidence/count, possible analytical impact, proposed treatment, validation check, and status. missingness, duplicates/key uni...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict how one extreme value can change mean, median, standard deviation, and a scatterplot. **Progressive hint:** Robust and non-robust summaries respond differently to outliers. **Verify:** Compute statistics before/after adding the extreme value; record the exact mean/median/std changes and describe the visible plot-scale effect.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict how one extreme value can change mean, median, standard deviation, and a scatterplot. Robust and non-robust summaries respond differently to outliers. Compute statistics...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace row grain from transaction-level data to a customer summary and explain which questions can no longer be answered afterward. **Progressive hint:** Aggregation discards within-customer event detail. **Verify:** List questions answerable at transaction grain, then assert the customer summary row count/uniqueness and identify at least one detail that cannot be recovered.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace row grain from transaction-level data to a customer summary and explain which questions can no longer be answered afterward. Aggregation discards within-customer event det...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement a compact profile returning shape, duplicate count, missing rates, numeric ranges, and unique counts. **Progressive hint:** Bound the result rather than dumping every row/value. **Verify:** Run the profile on ordinary, empty, duplicate, and missing fixtures; assert bounded keys/counts/rates without embedding full data values.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Implement a compact profile returning shape, duplicate count, missing rates, numeric ranges, and unique counts. Bound the result rather than dumping every row/value. Run the pro...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair an EDA that calculates correlations after target-derived fields were added and treats the strongest coefficient as causal. **Progressive hint:** Remove leakage and label correlations as associations. **Verify:** Remove the target-derived field, recompute the association, and label it noncausal; assert the leakage column cannot enter the reported matrix.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair an EDA that calculates correlations after target-derived fields were added and treats the strongest coefficient as causal. Remove leakage and label correlations as associ...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Handle constant, all-missing, and tiny-sample columns in plots and summaries; state which results are not meaningful. **Progressive hint:** A calculation returning a number does not guarantee interpretability. **Verify:** Detect constant/all-missing/tiny columns and assert each is skipped or annotated according to policy rather than reported as an interpretable statistic.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Handle constant, all-missing, and tiny-sample columns in plots and summaries; state which results are not meaningful. A calculation returning a number does not guarantee interpr...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
